# 医疗用户分析与风险评分系统 – 探索性分析

**Capstone Project | 2025.9 – 2025.12**

本 Notebook 对原始医疗数据进行探索性分析（EDA），帮助理解数据分布、缺失情况与目标变量关系。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'

# 加载数据
df = pd.read_csv('../data/raw/medical_users.csv')
print(f'数据规模：{df.shape[0]} 行 × {df.shape[1]} 列')
df.head()

In [ ]:
# 基础统计信息
print('=== 数据类型 ===')
print(df.dtypes)
print('\n=== 缺失值统计 ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print('\n=== 数值特征描述性统计 ===')
df.describe().round(2)

In [ ]:
# 目标变量分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(
    axes,
    ['churn_risk', 'revisit_probability'],
    ['Churn Risk Distribution', 'Revisit Probability Distribution']
):
    counts = df[col].value_counts()
    ax.bar(['Low Risk (0)', 'High Risk (1)'], counts.values, color=['#2ecc71', '#e74c3c'], alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, f'{v}\n({v/len(df):.1%})', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150)
plt.show()

In [ ]:
# 数值特征分布
num_cols = ['age', 'visit_count', 'total_medical_fee', 'last_visit_days_ago',
            'medication_adherence_score', 'followup_completion_rate']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), num_cols):
    ax.hist(df[col].dropna(), bins=30, color='#3498db', alpha=0.8, edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 相关性热图
corr_cols = ['age', 'visit_count', 'total_medical_fee', 'last_visit_days_ago',
             'medication_adherence_score', 'followup_completion_rate',
             'satisfaction_score', 'registration_months',
             'churn_risk', 'revisit_probability']

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 慢病与流失风险关系
churn_by_chronic = df.groupby('chronic_disease')['churn_risk'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(churn_by_chronic.index, churn_by_chronic.values,
              color='#e74c3c', alpha=0.8, edgecolor='white')
ax.set_title('Churn Risk Rate by Chronic Disease', fontsize=12, fontweight='bold')
ax.set_ylabel('Churn Risk Rate')
ax.set_xlabel('Chronic Disease Type')
ax.set_xticklabels(churn_by_chronic.index, rotation=15, ha='right')
for bar, val in zip(bars, churn_by_chronic.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/figures/churn_by_chronic.png', dpi=150)
plt.show()

In [ ]:
# 就诊费用箱线图（含异常值展示）
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot(df['total_medical_fee'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.7))
axes[0].set_title('Total Medical Fee (with outliers)', fontweight='bold')
axes[0].set_ylabel('Fee (CNY)')

# IQR 截断后
Q1 = df['total_medical_fee'].quantile(0.25)
Q3 = df['total_medical_fee'].quantile(0.75)
IQR = Q3 - Q1
fee_clipped = df['total_medical_fee'].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

axes[1].boxplot(fee_clipped.dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='#2ecc71', alpha=0.7))
axes[1].set_title('Total Medical Fee (after IQR clipping)', fontweight='bold')
axes[1].set_ylabel('Fee (CNY)')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Outlier Detection – Medical Fee (IQR Method)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/outlier_fee.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'IQR 截断前均值：{df["total_medical_fee"].mean():.0f}')
print(f'IQR 截断后均值：{fee_clipped.mean():.0f}')